In [ ]:
"""One thing about this Part is that by the time I reached here (Role Play Section) , I was able to understand how to build the solution for the problem given. 
So the last 4 boxes where completely typed by me, and if you check properly/compare with the structure of what the course gave me, its completely different from it."""

In [1]:
!pip install "ibm-watsonx-ai==1.0.8" --user
!pip install "langchain==0.2.11" --user
!pip install "langchain-ibm==0.1.7" --user
!pip install "langchain-core==0.2.43" --user

In [ ]:
import os
os._exit(00) #This is for restarting the kernel

In [1]:
# To suppress warnings generated in our code:
def warn(*args, **kwargs):
    pass
import warnings
warnings.warn = warn
warnings.filterwarnings('ignore')

# IBM WatsonX imports
from ibm_watsonx_ai.foundation_models import Model
from ibm_watsonx_ai.metanames import GenTextParamsMetaNames as GenParams
from ibm_watsonx_ai.foundation_models.utils.enums import ModelTypes

from langchain_ibm import WatsonxLLM
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableSequence
from langchain_core.messages import HumanMessage, SystemMessage
from langchain.chains import LLMChain  # Still using this for backward compatibility

In [2]:
def llm_model(prompt_txt, params=None):
    
    model_id = "ibm/granite-4-h-small"

    default_params = {
        "max_new_tokens": 256,
        "min_new_tokens": 100,
        "temperature": 0.5,
        "top_p": 0.2,
        "top_k": 1
    }

    url = "https://us-south.ml.cloud.ibm.com"
    project_id = "skills-network"
    
    granite_llm = WatsonxLLM(
        model_id=model_id,
        project_id=project_id,
        url=url,
        params=default_params
    )
    
    response = granite_llm.invoke(prompt_txt)
    return response

In [3]:
GenParams().get_example_values()

{'decoding_method': 'sample',
 'length_penalty': {'decay_factor': 2.5, 'start_index': 5},
 'temperature': 0.5,
 'top_p': 0.2,
 'top_k': 1,
 'random_seed': 33,
 'repetition_penalty': 2,
 'min_new_tokens': 50,
 'max_new_tokens': 200,
 'stop_sequences': ['fail'],
 ' time_limit': 600000,
 'truncate_input_tokens': 200,
 'prompt_variables': {'object': 'brain'},
 'return_options': {'input_text': True,
  'generated_tokens': True,
  'input_tokens': True,
  'token_logprobs': True,
  'token_ranks': False,
  'top_n_tokens': False}}

In [19]:
model_id = "ibm/granite-4-h-small"

parameters = {
    GenParams.MAX_NEW_TOKENS: 256,  # this controls the maximum number of tokens in the generated output
    GenParams.TEMPERATURE: 0.5, # this randomness or creativity of the model's responses: Like 0.9-> More creative, but may become less precise. 
    #While 0.1-> More precise, direct, but less creative
}

url = "https://us-south.ml.cloud.ibm.com"
project_id = "skills-network"
#Connecting with the AI
llm = WatsonxLLM(
        model_id=model_id,
        url=url,
        project_id=project_id,
        params=parameters
    )
llm

WatsonxLLM(model_id='ibm/granite-4-h-small', project_id='skills-network', url=SecretStr('**********'), apikey=SecretStr('**********'), params={'max_new_tokens': 256, 'temperature': 0.5}, watsonx_model=<ibm_watsonx_ai.foundation_models.inference.model_inference.ModelInference object at 0x7816a7891700>)

In [22]:
role = """
    Product review analyzer
"""

categories = "Positive, Negative, Neutral"
features = "What is the quality of the product?"
template = """
    You are an expert {role}. I have this review: {reviews}. Please classify it according to these sentiments: {categories}.
    After Classifying them accordingly, please provide me with the features {features} of the product.
    Also, I have a question {question} about this product. Please answer it.  
    Now, summarize the review {reviews} in one simple sentence.
    Answer:

"""
prompt = PromptTemplate.from_template(template)
prompt 
#So let,s first understand the conditions of the questions given to me (3):
# Identify the sentiment (positive, negative, or neutral), Extract mentioned product features, Provide a one-sentence summary of the review.
#So what happened here was that firstly, I made the template of my prompt template, right? And then whatever I needed to put inside the where given on top.
#Then I put the template as the prompt.

PromptTemplate(input_variables=['categories', 'features', 'question', 'reviews', 'role'], template='\n    You are an expert {role}. I have this review: {reviews}. Please classify it according to these sentiments: {categories}.\n    After Classifying them accordingly, please provide me with the features {features} of the product.\n    Also, I have a question {question} about this product. Please answer it.  \n    Now, summarize the review {reviews} in one simple sentence.\n    Answer:\n\n')

In [23]:
from langchain_core.runnables import RunnableLambda

# Define a function to ensure proper formatting
#Building the Chain
def format_prompt(variables):
    return prompt.format(**variables)

In [ ]:
# Create the LCEL chain
classification_chain = (
    RunnableLambda(format_prompt)
    | llm 
    | StrOutputParser()
)
reviews = [
    "I love this smartphone! The camera quality is exceptional and the battery lasts all day. The only downside is that it heats up a bit during gaming.",
    "This laptop is terrible. It's slow, crashes frequently, and the keyboard stopped working after just two months. Customer service was unhelpful."
]

while True:
    question = input("Question: ")
    if question.lower() in ["quit", "exit", "bye"]:
        print("Answer: Goodbye!")
        break
    
    for i, review in enumerate(reviews):
        print(f"==== Review #{i+1} ====")
        category = classification_chain.invoke({"role": role,"reviews": review,  # single review, not all of them
        "categories": categories, "features": features, "question": question})
        print(category)
        print()
#Now, I just passed the reviews, and aksed a question, and done.

Question:  Should I get this product?


==== Review #1 ====
Positive

Features of the product:
- Exceptional camera quality
- Long-lasting battery life
- Good performance for daily use

Should you get this product?
If you prioritize camera quality, battery life, and overall performance for daily use, this smartphone is a great choice. However, if you frequently engage in intensive gaming sessions, you might want to consider a device with better heat management.

Summary:
The reviewer loves the smartphone's exceptional camera quality, long battery life, and good performance, but notes that it heats up during gaming.

==== Review #2 ====
Based on the content of the review, it can be classified as Negative.

The review mentions several negative aspects of the product:
- The laptop is slow
- It crashes frequently
- The keyboard stopped working after just two months
- Customer service was unhelpful

These issues indicate poor quality and performance of the laptop, as well as unsatisfactory customer support.

Given the negative se